# Notebook 2: Accessing Real DFT Data from the Materials Project
Prepared by Festus Ogungbemiro

## Overview
In this notebook, we will connect to the **Materials Project** (MP) database using its REST API. The Materials Project provides computed properties (DFT) for over 140,000 inorganic materials. We will query for materials containing elements known for antimicrobial activity (Ag, Cu, Zn, Ti, Au, Mg, Fe) and retrieve key properties such as band gap, formation energy, density, and volume. This raw data will be saved for later feature engineering and machine learning.

**Key steps:**
1. Install and import required libraries.
2. Securely set up your Materials Project API key.
3. Test the connection with a simple query.
4. Construct a query to fetch materials containing our target elements (with pagination to get all results).
5. Parse the results into a Pandas DataFrame.
6. Save the data to a CSV file for use in subsequent notebooks.

---
## 1. Installing Required Libraries

If you haven't already, install the necessary libraries. Run the following cell (uncomment the lines).

In [1]:
!pip install mp-api pymatgen pandas numpy

---
## 2. Importing Libraries

We will import the libraries we need. `MPRester` is the main class for interacting with the Materials Project API.

In [2]:
import os
import pandas as pd
import numpy as np
from mp_api.client import MPRester

---
## 3. Setting Up Your API Key

### 3.1 Get Your API Key
If you don't have an API key, sign up for a free account at [Materials Project](https://materialsproject.org/) and navigate to your dashboard to generate a key.

### 3.2 Secure Storage
Hard‑coding API keys in a notebook is a security risk. Instead, we will read the key from an environment variable. You can set it in your terminal or add it to a `.env` file. For testing purposes, we provide a commented line where you can paste your key – **but remember to remove it before sharing the notebook**.

In [3]:
# Option 1: Set environment variable (recommended for security)
# In your terminal, run: export MP_API_KEY="your_key_here"
# Then in Python:
API_KEY = os.getenv("MP_API_KEY")

# Option 2: For testing only – paste your key directly (UNCOMMENT AND REPLACE)
API_KEY = "XsptzNvWVc8j1VeOtgTuqiHwZz1whMmk"

if API_KEY is None:
    raise ValueError("API key not found. Set the MP_API_KEY environment variable or uncomment the line with your key.")
else:
    print("API key loaded successfully.")

API key loaded successfully.


---
## 4. Testing the Connection

Let's perform a simple test to ensure our connection works. We'll fetch the record for a known material, e.g., silver (mp-124).

In [6]:
# Test the connection by fetching a known material (mp-124) using search
with MPRester(API_KEY) as mpr:
    docs = mpr.materials.summary.search(
        material_ids=["mp-124"],
        fields=["material_id", "formula_pretty", "band_gap",
                "formation_energy_per_atom", "density", "volume"]
    )
    if docs:
        doc = docs[0]
        print("Material ID:", doc.material_id)
        print("Formula:", doc.formula_pretty)
        print("Band gap (eV):", doc.band_gap)
        print("Formation energy per atom (eV):", doc.formation_energy_per_atom)
        print("Density (g/cm³):", doc.density)
        print("Volume (Å³):", doc.volume)
    else:
        print("No material found with ID mp-124.")

Retrieving SummaryDoc documents:   0%|          | 0/1 [00:00<?, ?it/s]

Material ID: mp-124
Formula: Ag
Band gap (eV): 0.0
Formation energy per atom (eV): 0.0021267149999990003
Density (g/cm³): 10.362566636379526
Volume (Å³): 17.28523120082949


**Explanation**:
- `MPRester` is used as a context manager (`with` statement) to automatically close the connection.
- `get_data_by_id()` retrieves a single material document.
- We print some of its attributes to confirm we received data.

---
## 5. Defining Target Elements

Based on literature, elements like silver (Ag), copper (Cu), zinc (Zn), titanium (Ti), gold (Au), magnesium (Mg), and iron (Fe) are known for antimicrobial properties. We will query for materials containing any of these elements.

In [7]:
target_elements = ["Ag", "Cu", "Zn", "Ti", "Au", "Mg", "Fe"]
print("Target elements:", target_elements)

Target elements: ['Ag', 'Cu', 'Zn', 'Ti', 'Au', 'Mg', 'Fe']


---
## 6. Fetching Data for All Target Elements

We will loop over each element and retrieve **all** materials that contain that element. To avoid duplicates (materials containing multiple target elements), we will store unique material IDs.

The fields we want:
- `material_id`
- `formula_pretty`
- `band_gap`
- `formation_energy_per_atom`
- `density`
- `volume`
- `is_metal`
- `nelements`

We'll accumulate results in a list of dictionaries and then convert to a DataFrame.

**Important**: To ensure we get **all** materials (not just the first 100), we set `num_chunks=None` in the search call. This tells the API to paginate through all available results.

In [8]:
all_data = []
seen_mpids = set()

with MPRester(API_KEY) as mpr:
    for el in target_elements:
        print(f"Querying materials containing {el}...")
        docs = mpr.materials.summary.search(
            elements=[el],
            fields=[
                "material_id",
                "formula_pretty",
                "band_gap",
                "formation_energy_per_atom",
                "density",
                "volume",
                "is_metal",
                "nelements"
            ],
            num_chunks=None  # fetch all pages (no limit)
        )
        
        for doc in docs:
            mpid = str(doc.material_id)
            if mpid not in seen_mpids:
                all_data.append({
                    "MPID": mpid,
                    "Formula": doc.formula_pretty,
                    "Band_Gap_eV": doc.band_gap,
                    "Formation_Energy_eV_atom": doc.formation_energy_per_atom,
                    "Density_g_cm3": doc.density,
                    "Volume_A3": doc.volume,
                    "Is_Metal": doc.is_metal,
                    "Num_Elements": doc.nelements
                })
                seen_mpids.add(mpid)
        
        print(f"  -> Cumulative unique records: {len(all_data)}")

print(f"\nTotal unique materials fetched: {len(all_data)}")

Querying materials containing Ag...


Retrieving SummaryDoc documents:   0%|          | 0/4129 [00:00<?, ?it/s]

  -> Cumulative unique records: 4129
Querying materials containing Cu...


Retrieving SummaryDoc documents:   0%|          | 0/9856 [00:00<?, ?it/s]

  -> Cumulative unique records: 13853
Querying materials containing Zn...


...

  -> Cumulative unique records: 29431
Querying materials containing Mg...


Retrieving SummaryDoc documents:   0%|          | 0/19098 [00:00<?, ?it/s]

  -> Cumulative unique records: 44994
Querying materials containing Fe...


Retrieving SummaryDoc documents:   0%|          | 0/12956 [00:00<?, ?it/s]

  -> Cumulative unique records: 55200

Total unique materials fetched: 55200


**Explanation**:
- `mpr.materials.summary.search()` allows us to search for materials matching criteria. Here we search for any material that contains the element `el`.
- We specify the fields we want to retrieve to minimise data transfer.
- `num_chunks=None` ensures we get all results (the API paginates automatically).
- The loop collects documents, and we use a set `seen_mpids` to avoid adding the same material multiple times if it contains more than one target element.

---
## 7. Creating a DataFrame

Now we convert our list of dictionaries into a Pandas DataFrame for easier manipulation.

In [9]:
df = pd.DataFrame(all_data)
print(f"DataFrame shape: {df.shape}")
df.head()

DataFrame shape: (55200, 8)


,MPID,Formula,Band_Gap_eV,Formation_Energy_eV_atom,Density_g_cm3,Volume_A3,Is_Metal,Num_Elements
0,mp-989737,Ag,0.0,0.010382,9.922633,54.154786,True,1
1,mp-10597,Ag,0.0,0.007669,10.302467,34.772131,True,1
2,mp-8566,Ag,0.0,0.000000,10.315418,69.456950,True,1
3,mp-124,Ag,0.0,0.002127,10.362567,17.285231,True,1
4,mp-2646971,Ag,0.0,0.034337,9.866742,18.153850,True,1


---
## 8. Saving Raw Data to CSV

We save the DataFrame to a CSV file. This file will be used in the next notebook for feature engineering and analysis.

In [10]:
output_file = "01_materials_project_raw_data.csv"
df.to_csv(output_file, index=False)
print(f"Data saved to {output_file}")

Data saved to 01_materials_project_raw_data.csv


---
## 9. Quick Data Inspection

Let's take a quick look at the data we've collected: basic statistics and some example entries.

In [11]:
print("DataFrame info:")
df.info()

print("\nSummary statistics:")
df.describe()

print("\nFirst 5 rows:")
df.head()

DataFrame info:
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 55200 entries, 0 to 55199
Data columns (total 8 columns):
 #   Column                    Non-Null Count  Dtype  
---  ------                    --------------  -----  
 0   MPID                      55200 non-null  object 
 1   Formula                   55200 non-null  object 
 2   Band_Gap_eV               55200 non-null  float64
 3   Formation_Energy_eV_atom  55200 non-null  float64
 4   Density_g_cm3             55200 non-null  float64
 5   Volume_A3                 55200 non-null  float64
 6   Is_Metal                  55200 non-null  bool   
 7   Num_Elements              55200 non-null  int64  
dtypes: bool(1), float64(4), int64(1), object(2)
memory usage: 3.0+ MB

Summary statistics:

First 5 rows:


,MPID,Formula,Band_Gap_eV,Formation_Energy_eV_atom,Density_g_cm3,Volume_A3,Is_Metal,Num_Elements
0,mp-989737,Ag,0.0,0.010382,9.922633,54.154786,True,1
1,mp-10597,Ag,0.0,0.007669,10.302467,34.772131,True,1
2,mp-8566,Ag,0.0,0.000000,10.315418,69.456950,True,1
3,mp-124,Ag,0.0,0.002127,10.362567,17.285231,True,1
4,mp-2646971,Ag,0.0,0.034337,9.866742,18.153850,True,1


---
## Summary
In this notebook, we successfully:
- Connected to the Materials Project API.
- Queried for materials containing antimicrobial elements, fetching **all** results via pagination.
- Stored the raw data in a structured CSV file.

The next notebook (**Notebook 3**) will focus on feature engineering using `pymatgen` and exploratory data analysis. You will load the CSV file saved here and enrich it with elemental descriptors.